# Steering — DiffAware / CtxtAware (instruct models)

Reproduces **Table 3** of the paper: DiffAware and CtxtAware under B→item
attention steering, following the exact Wang et al. protocol
(generation-based evaluation, refusal filtering, cluster bootstrap CI over
unique scenarios), plus the **A→item steering control** (non-associated
identity).

Outputs, written under `./results/<model>_instruct/`:
- `diffaware_results.pkl`: metrics per α, B→item steering
- `diffaware_control_results.pkl`: metrics per α, A→item control
- `diffaware_steering.png`: DiffAware / CtxtAware / accuracy vs α

The three helper functions (`build_wang_positions`, `generate_pipeline`,
`compute_diffaware_wang`) live in `common/instruct/steering.py`.

Note: paths are anchored on the repo root by `common/config.py`, so the
notebook runs regardless of the kernel working directory.


In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma2", "nemo"}

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED

import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

from common.text_parsers import norm_identity, extract_options, parse_meta
from common.instruct.data import load_n4, extract_neutral_item
from common.instruct.steering import (build_wang_positions, generate_pipeline,
                                      compute_diffaware_wang)

# Local aliases used by the cells below
DATA_DIR = config.DATA_DIR
OUTPUT_DIR = config.OUTPUT_DIR
HF_TOKEN = config.HF_TOKEN
MC = config.MC

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

## Load data and model

In [ ]:
# Load data
cultural_items, neutral_items = load_n4(DATA_DIR)

# Load model
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

first_device = next(model.parameters()).device

# Publish runtime singletons for the common helpers (repo convention)
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

In [ ]:
FINAL_HEADS = CFG['heads']  # override here if CV discovered different heads
print(f"  Heads: {FINAL_HEADS}")

## Stage 5 — DiffAware steering (Wang et al. protocol, with bootstrap CI)

In [ ]:
# ================================================================
# DIFFAWARE STEERING: generation-based, Wang exact protocol
# ================================================================

print(f"\n{'='*80}")
print(f"STAGE 5: DIFFAWARE STEERING — {CFG['label']}")
print(f"{'='*80}")

# Build neq/eq data from raw N4
neq_data, eq_data = [], []

for item in cultural_items:
    q, ans, meta = item
    try:
        qtype, c_item, in_groups, out_groups = parse_meta(meta)
    except: continue
    oa, ob = extract_options(q)
    if oa is None: continue
    assoc_pos = None
    for g in in_groups:
        if norm_identity(g) == norm_identity(oa):
            assoc_pos = 'a'; break
        elif norm_identity(g) == norm_identity(ob):
            assoc_pos = 'b'; break
    if assoc_pos is None: continue
    neq_data.append({
        'prompt': q + '\n\n' + MC, 'ans': int(ans), 'item': c_item,
        'assoc_pos': assoc_pos, 'oa': oa, 'ob': ob,
    })

for item in neutral_items:
    q, ans, meta = item
    try:
        qtype, c_item, in_groups, out_groups = parse_meta(meta)
    except: continue
    oa, ob = extract_options(q)
    if oa is None: continue
    assoc_pos = None
    for g in in_groups:
        if norm_identity(g) == norm_identity(oa):
            assoc_pos = 'a'; break
        elif norm_identity(g) == norm_identity(ob):
            assoc_pos = 'b'; break
    if assoc_pos is None: continue
    n_item = extract_neutral_item(q, qtype)
    eq_data.append({
        'prompt': q + '\n\n' + MC, 'ans': int(ans), 'item': c_item,
        'assoc_pos': assoc_pos, 'oa': oa, 'ob': ob,
        'neutral_item': n_item,
    })

print(f"  neq (match): {len(neq_data)}, eq (mismatch): {len(eq_data)}")

# Build positions for neq/eq
print("  Building neq positions...")
neq_positions = build_wang_positions(neq_data, tokenizer)
print("  Building eq positions...")
eq_positions = build_wang_positions(eq_data, tokenizer)

# Pipeline
import transformers
pipe = transformers.pipeline(
    "text-generation", model=model, tokenizer=tokenizer, device_map="auto")
if pipe.model.config.pad_token_id is None:
    pipe.model.config.pad_token_id = pipe.model.config.eos_token_id

# Run steering
DA_ALPHAS = [0.0, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
da_results = {}

for alpha in DA_ALPHAS:
    print(f"\n  α = {alpha:.1f} ...")
    neq_gen = generate_pipeline(pipe, neq_data, neq_positions, FINAL_HEADS, alpha)
    eq_gen = generate_pipeline(pipe, eq_data, eq_positions, FINAL_HEADS, alpha)
    metrics = compute_diffaware_wang(neq_gen, neq_data, eq_gen, eq_data)
    da_results[alpha] = metrics
    d = metrics['diffaware']
    c = metrics['ctxtaware']
    print(f"    DiffAware: {d[1]:.3f} [{d[0]:.3f}, {d[2]:.3f}]")
    print(f"    CtxtAware: {c[1]:.3f} [{c[0]:.3f}, {c[2]:.3f}]")
    print(f"    ≠ acc: {metrics['neq_acc']:.3f}  = acc: {metrics['eq_acc']:.3f}")
    torch.cuda.empty_cache()

# Table
print(f"\n{'α':>6s}  {'DiffAware':>10s}  {'CtxtAware':>10s}  {'≠ acc':>7s}  {'= acc':>7s}")
print(f"{'─'*6}  {'─'*10}  {'─'*10}  {'─'*7}  {'─'*7}")
for a in sorted(da_results.keys()):
    r = da_results[a]
    print(f"{a:6.1f}  {r['diffaware'][1]:10.3f}  {r['ctxtaware'][1]:10.3f}  "
          f"{r['neq_acc']:7.3f}  {r['eq_acc']:7.3f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
alphas_sorted = sorted(da_results.keys())
diff_vals = [da_results[a]['diffaware'] for a in alphas_sorted]
ctxt_vals = [da_results[a]['ctxtaware'] for a in alphas_sorted]
eq_accs = [da_results[a]['eq_acc'] for a in alphas_sorted]
neq_accs = [da_results[a]['neq_acc'] for a in alphas_sorted]

ax = axes[0]
ax.plot(alphas_sorted, [d[1] for d in diff_vals], 's-', color='#d62728', linewidth=2, markersize=8)
ax.fill_between(alphas_sorted, [d[0] for d in diff_vals], [d[2] for d in diff_vals],
                color='#d62728', alpha=0.15)
ax.set_xlabel('α'); ax.set_ylabel('DiffAware')
ax.set_title(f'{CFG["label"]}: DiffAware vs α'); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(alphas_sorted, [c[1] for c in ctxt_vals], 'o-', color='#1f77b4', linewidth=2, markersize=8)
ax.fill_between(alphas_sorted, [c[0] for c in ctxt_vals], [c[2] for c in ctxt_vals],
                color='#1f77b4', alpha=0.15)
ax.set_xlabel('α'); ax.set_ylabel('CtxtAware')
ax.set_title(f'{CFG["label"]}: CtxtAware vs α'); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(alphas_sorted, neq_accs, 's-', color='#d62728', label='≠ acc')
ax.plot(alphas_sorted, eq_accs, 'o-', color='#1f77b4', label='= acc')
ax.set_xlabel('α'); ax.set_ylabel('Accuracy')
ax.set_title(f'{CFG["label"]}: Accuracy vs α'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "diffaware_steering.png", dpi=150)
plt.show()

# Save all results
with open(OUTPUT_DIR / "diffaware_results.pkl", "wb") as f:
    pickle.dump(da_results, f)

## Stage 6 — Control: A→item steering (non-associated identity)

In [ ]:
# ================================================================
# CONTROL: A→item STEERING
# ================================================================

print(f"\n{'='*80}")
print(f"STAGE 6: CONTROL — A→item STEERING — {CFG['label']}")
print(f"{'='*80}")

DA_ALPHAS_CTRL = [0.0, 1.0, 2.0, 3.0, 5.0]
da_ctrl_results = {}

for alpha in DA_ALPHAS_CTRL:
    print(f"\n  α = {alpha:.1f} (A→item) ...")
    neq_gen = generate_pipeline(pipe, neq_data, neq_positions, FINAL_HEADS, alpha,
                                 steer_target='A')
    eq_gen = generate_pipeline(pipe, eq_data, eq_positions, FINAL_HEADS, alpha,
                                steer_target='A')
    metrics = compute_diffaware_wang(neq_gen, neq_data, eq_gen, eq_data)
    da_ctrl_results[alpha] = metrics
    d = metrics['diffaware']
    c = metrics['ctxtaware']
    print(f"    DiffAware: {d[1]:.3f} [{d[0]:.3f}, {d[2]:.3f}]")
    print(f"    CtxtAware: {c[1]:.3f} [{c[0]:.3f}, {c[2]:.3f}]")
    torch.cuda.empty_cache()

# Comparison table
print(f"\n{'─'*70}")
print(f"COMPARISON: B→item (experimental) vs A→item (control)")
print(f"{'─'*70}")
print(f"{'α':>6s}  {'B→i DA':>8s}  {'A→i DA':>8s}  {'Δ(DA)':>8s}  │  "
      f"{'B→i CA':>8s}  {'A→i CA':>8s}  {'Δ(CA)':>8s}")
print(f"{'─'*6}  {'─'*8}  {'─'*8}  {'─'*8}  │  {'─'*8}  {'─'*8}  {'─'*8}")
for a in sorted(set(DA_ALPHAS) & set(DA_ALPHAS_CTRL)):
    rb = da_results[a]
    rc = da_ctrl_results[a]
    d_da = rb['diffaware'][1] - rc['diffaware'][1]
    d_ca = rb['ctxtaware'][1] - rc['ctxtaware'][1]
    print(f"{a:6.1f}  {rb['diffaware'][1]:8.3f}  {rc['diffaware'][1]:8.3f}  {d_da:+8.3f}  │  "
          f"{rb['ctxtaware'][1]:8.3f}  {rc['ctxtaware'][1]:8.3f}  {d_ca:+8.3f}")

# Save control results
with open(OUTPUT_DIR / "diffaware_control_results.pkl", "wb") as f:
    pickle.dump(da_ctrl_results, f)